## 1. Bibliotecas

In [1]:
import os
import re
import pandas as pd

## 2. Caminhos dos arquivos

Caminhos para os arquivos de dados brutos do Fake.Br e do Fake.Br-LLM.

In [2]:
FAKEBR_PATH = '../data/raw/Fake_Br/full_texts/fake'

In [3]:
TRAIN_FAKEBR_LLM_PATH = '../data/raw/Fake.Br-LLM/train-fake-LLM'
TEST_FAKEBR_LLM_PATH = '../data/raw/Fake.Br-LLM/test-fake-LLM'

## 3. Funções de carregamento

Funções para ler e extrair o conteúdo dos arquivos de cada fonte de dados.

In [4]:
def read_news_fakebr(file_path):
  """
  Lê um arquivo de notícia falsa do Fake.Br Corpus.

  Parâmetros:
    file_path (str): caminho completo do arquivo .txt

  Retorna:
    dict com 'id' (extraído do nome do arquivo) e 'text' (conteúdo do arquivo)
  """

  with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()
    
  file_name = os.path.basename(file_path)
  id_news = os.path.splitext(file_name)[0]
  return {'id': id_news, 'text': text.strip()}

In [5]:
def read_news_fakebr_llm(file_path):
  with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

  file_name = os.path.basename(file_path)
  id_news = os.path.splitext(file_name)[0]

  match_original = re.search('<originalText>(.*?)</originalText>', text, re.DOTALL)
  match_synthetic = re.search('<syntheticText>(.*?)</syntheticText>', text, re.DOTALL)
  match_changes = re.search('<changes>(.*?)</changes>', text, re.DOTALL)

  original_news = match_original.group(1).strip() if match_original else ''
  synthetic_news = match_synthetic.group(1).strip() if match_synthetic else ''
  llm_changes = match_changes.group(1).strip() if match_changes else ''

  return {'id': id_news, 'original_news': original_news, 'synthetic_news': synthetic_news, 'llm_changes': llm_changes}

## 4. Carregamento dos dados

Aplicação das funções de leitura sobre todos os arquivos de cada pasta, gerando as listas de notícias.

In [6]:
human_fakebr_news_raw = []

for file_name in os.listdir(FAKEBR_PATH):
  full_path = os.path.join(FAKEBR_PATH, file_name)
  news = read_news_fakebr(full_path)
  human_fakebr_news_raw.append(news)

print(f'Total de notícias carregadas: {len(human_fakebr_news_raw)}')

Total de notícias carregadas: 3600


In [7]:
train_fakebr_llm_raw = []

for file_name in os.listdir(TRAIN_FAKEBR_LLM_PATH):
  full_path = os.path.join(TRAIN_FAKEBR_LLM_PATH, file_name)
  synthetic_news = read_news_fakebr_llm(full_path)
  train_fakebr_llm_raw.append(synthetic_news)

print(f'Total de notícias para treinamento carregadas: {len(train_fakebr_llm_raw)}')

Total de notícias para treinamento carregadas: 2880


In [8]:
test_fakebr_llm_raw = []

for file_name in os.listdir(TEST_FAKEBR_LLM_PATH):
  full_path = os.path.join(TEST_FAKEBR_LLM_PATH, file_name)
  synthetic_news = read_news_fakebr_llm(full_path)
  test_fakebr_llm_raw.append(synthetic_news)

print(f'Total de notícias para teste carregadas: {len(test_fakebr_llm_raw)}')

Total de notícias para teste carregadas: 720


## 5. Consolidação

União das notícias humanas e sintéticas em um único DataFrame, seguindo a estrutura de colunas definida (id, text, label, origin, source, split, llm_explanation).

In [9]:
train_ids = set()
test_ids = set()

for item in train_fakebr_llm_raw:
  train_ids.add(item['id'])

for item in test_fakebr_llm_raw:
  test_ids.add(item['id'])

In [10]:
rows_fakebr = []

for news in human_fakebr_news_raw:
  if news['id'] in train_ids:
    split = 'train'
  elif news['id'] in test_ids:
    split = 'test'
  else:
    split = None

  rows_fakebr.append({
    'id': news['id'],
    'text': news['text'],
    'label': 'fake',
    'origin': 'human',
    'source': 'fake.br',
    'split': split,
    'llm_explanation': None
  })

In [11]:
rows_fakebr_llm = []

for item in train_fakebr_llm_raw:
  rows_fakebr_llm.append({
    'id': item['id'], 'text': item['original_news'],
    'label': 'true', 'origin': 'human', 'source': 'fake.br-llm',
    'split': 'train', 'llm_explanation': None
  })

  rows_fakebr_llm.append({
    'id': item['id'], 'text': item['synthetic_news'],
    'label': 'fake', 'origin': 'synthetic', 'source': 'fake.br-llm',
    'split': 'train', 'llm_explanation': item['llm_changes']
  })

for item in test_fakebr_llm_raw:
  rows_fakebr_llm.append({
    'id': item['id'], 'text': item['original_news'],
    'label': 'true', 'origin': 'human', 'source': 'fake.br-llm',
    'split': 'test', 'llm_explanation': None
  })

  rows_fakebr_llm.append({
    'id': item['id'], 'text': item['synthetic_news'],
    'label': 'fake', 'origin': 'synthetic', 'source': 'fake.br-llm',
    'split': 'test', 'llm_explanation': item['llm_changes']
  })

## 6. Validação do DataFrame

Conferência de contagens e amostras do DataFrame final para garantir que os dados foram carregados corretamente.

In [12]:
df_news = pd.DataFrame(rows_fakebr + rows_fakebr_llm)

print(f'Total de linhas: {len(df_news)}')
print(df_news['label'].value_counts())
print(df_news['origin'].value_counts())

Total de linhas: 10800
label
fake    7200
true    3600
Name: count, dtype: int64
origin
human        7200
synthetic    3600
Name: count, dtype: int64


In [13]:
df_news.head()

,id,text,label,origin,source,split,llm_explanation
0,1,Kátia Abreu diz que vai colocar sua expulsão e...,fake,human,fake.br,train,NaN
1,10,"Dr. Ray peita Bolsonaro, chama-o de conservad...",fake,human,fake.br,train,NaN
2,100,Reinaldo Azevedo desmascarado pela Polícia Fed...,fake,human,fake.br,train,NaN
3,1000,Relatório assustador do BNDES mostra dinheiro ...,fake,human,fake.br,train,NaN
4,1001,"Radialista americano fala sobre o PT: ""Eles ve...",fake,human,fake.br,test,NaN


In [14]:
df_news.tail()

,id,text,label,origin,source,split,llm_explanation
10795,980,Exclusivo: Jovem vítima de conspiração crimino...,fake,synthetic,fake.br-llm,test,1. **Adição de Elemento Conspiratório**: Intro...
10796,985,2a Fase da Operação Ouro Verde cumpre mandados...,true,human,fake.br-llm,test,NaN
10797,985,Manchetes chocantes revelam o escândalo da Ope...,fake,synthetic,fake.br-llm,test,1. **Exagero de Fatos:** Aumentei a quantia de...
10798,991,Por que achamos que o mundo está pior do que r...,true,human,fake.br-llm,test,NaN
10799,991,Pesquisa revela: O mundo está realmente pioran...,fake,synthetic,fake.br-llm,test,1. **Taxa de Homicídios**: Alterei a afirmação...


## 7. Salvamento do DataFrame

Exportação do DataFrame consolidado para uso nos notebooks seguintes.

In [15]:
os.makedirs('../data/processed', exist_ok=True)
df_news.to_csv('../data/processed/news_dataset.csv', index=False)